# 1. Data Loading and Preprocessing

**Project Title:** AI for low‑carbon energy scheduling: forecasting electricity carbon intensity and recommending cleaner time windows

**Purpose:** This notebook handles the ingestion of the London Smart Meter dataset and the engineering of features required to forecast Grid Carbon Intensity. 

**Course Reference:** Data loading and cleaning techniques are adapted from **Lab 01 & 02 (AI & Sustainability)**, focusing on handling CSV structures and datetime parsing for time-series sustainability data.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

# --- Configuration ---
# DATA_DIR: Location of the 112 CSV blocks from the London Smart Meter dataset (Kaggle)
DATA_DIR = 'archive/halfhourly_dataset/halfhourly_dataset/'

# EMISSIONS_DATA_PATH: Placeholder for external Grid Carbon Intensity data
EMISSIONS_DATA_PATH = 'data/raw/grid_emission_factor.csv'

# PROCESSED_DATA_DIR: Output folder for cleaned datasets
PROCESSED_DATA_DIR = 'data/processed/'

# NUM_BLOCKS_TO_LOAD: Number of households blocks to aggregate. 
# Analysis in this project covers 5, 30, and 60 blocks to evaluate Sustainable AI trade-offs.
NUM_BLOCKS_TO_LOAD = 60 

if not os.path.exists(PROCESSED_DATA_DIR):
    os.makedirs(PROCESSED_DATA_DIR)

## 1.1 Loading and Aggregating Smart Meter Data

**Methodology:** We aggregate individual household data into a system-wide demand signal. This reduces noise and provides a better proxy for total grid load, which is a primary driver of carbon intensity.

**Ref:** Block loading logic inspired by standard Python `glob` patterns for large-scale environmental datasets.

In [ ]:
def load_demand_features(directory, num_blocks=60):
    """
    Iterates through CSV blocks, cleans energy values, and aggregates by timestamp.
    """
    all_files = sorted(glob.glob(os.path.join(directory, 'block_*.csv')))
    files_to_load = all_files[:num_blocks]
    aggregated_series = None
    
    for f in files_to_load:
        # DtypeWarning handling: Some CSVs have mixed types (e.g., 'Null' strings in energy)
        df = pd.read_csv(f, parse_dates=['tstp'], low_memory=False)
        df.rename(columns={'tstp': 'timestamp', 'energy(kWh/hh)': 'energy'}, inplace=True)
        
        # Preprocessing: Convert energy to numeric, dropping invalid rows (Lab 01 technique)
        df['energy'] = pd.to_numeric(df['energy'], errors='coerce')
        df.dropna(subset=['energy', 'timestamp'], inplace=True)
        
        # Aggregation: Sum energy across all households in the block for each half-hour slot
        block_agg = df.groupby('timestamp')['energy'].sum()
        
        if aggregated_series is None:
            aggregated_series = block_agg
        else:
            aggregated_series = aggregated_series.add(block_agg, fill_value=0)
            
    # Return a regularized 30-minute time series (Lab 03 technique for signal consistency)
    return aggregated_series.to_frame().resample('30T').sum()

demand_df = load_demand_features(DATA_DIR, num_blocks=NUM_BLOCKS_TO_LOAD)

# --- Integrate Carbon Intensity (Target Variable) ---
if os.path.exists(EMISSIONS_DATA_PATH):
    ci_df = pd.read_csv(EMISSIONS_DATA_PATH, parse_dates=['datetime'])
    ci_df.rename(columns={'datetime': 'timestamp'}, inplace=True)
    ci_df.set_index('timestamp', inplace=True)
    final_df = demand_df.join(ci_df, how='inner')
else:
    # Synthetic Target Model: 
    # Modeled with a diurnal sine wave + a demand-driven linear component. 
    # Carbon intensity typically peaks when demand is highest (peaker plants).
    final_df = demand_df.copy()
    hour = final_df.index.hour
    final_df['carbon_intensity'] = 150 + 50 * np.sin(2 * np.pi * (hour - 4) / 24) + \
                                    0.2 * (final_df['energy'] / final_df['energy'].max() * 100) + \
                                    np.random.normal(0, 5, len(final_df))

print(f"Dataset ready with {len(final_df)} samples. Target: carbon_intensity")

## 1.2 Feature Engineering for Intensity Forecasting

**Objective:** Create predictive features that capture the autocorrelative nature of grid intensity.

**Lab Reference:** Autoregressive lag features and rolling averages are standard practices in **Lab 04 (DL4TS)** for time-series sustainability modeling.

In [ ]:
def engineer_intensity_features(df):
    df = df.copy()
    
    # Calendar Features (Diurnal and Weekly cycles)
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    
    # Autoregressive (AR) Features: Previous values of Carbon Intensity
    df['intensity_lag_30m'] = df['carbon_intensity'].shift(1)
    df['intensity_lag_1h'] = df['carbon_intensity'].shift(2)
    df['intensity_lag_24h'] = df['carbon_intensity'].shift(48)
    
    # Moving Average: Smoothed trend of grid cleanliness
    df['intensity_rolling_mean_6h'] = df['carbon_intensity'].shift(1).rolling(window=12).mean()
    
    # Exogenous Feature: Most recent system load as a leading indicator
    df['demand_lag_30m'] = df['energy'].shift(1)
    
    df.dropna(inplace=True)
    return df

processed_df = engineer_intensity_features(final_df)
print(f"Final feature set shape: {processed_df.shape}")

## 1.3 Splitting and Saving

**Validation Strategy:** We use a chronological split (70/15/15) instead of a random shuffle to respect the temporal causality of time-series data (Lab 04 approach).

In [ ]:
train_size = int(len(processed_df) * 0.7)
val_size = int(len(processed_df) * 0.15)

train_df = processed_df.iloc[:train_size]
val_df = processed_df.iloc[train_size:train_size+val_size]
test_df = processed_df.iloc[train_size+val_size:]

# Save as CSV for transparency and reproducibility
train_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'train_ci.csv'))
val_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'val_ci.csv'))
test_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'test_ci.csv'))

print("Datasets for Carbon Intensity forecasting successfully saved.")